# 🔌 MCP & Tooling: Zero to Hero — A Guided Lab

The **Model Context Protocol (MCP)** is an open standard that lets LLM applications connect to
external tools, data, and prompts through a **uniform interface** — like "USB-C for AI tools."
Instead of hand-wiring every integration, you expose capabilities via an MCP **server** and any
MCP **client** (Claude Desktop, IDEs, agents) can use them.

**Runs 100% offline.** We build a minimal in-memory MCP server & client with the *same
primitives* as the real protocol (tools, resources, prompts) so the concepts transfer directly.

**How this lab works** — 📖 Theory → 🧠 Mental model → 🖼️ ASCII diagram → 🔬 Example →
⚡ Pro tips → ⚠️ Traps → ✏️ Your Turn → ✅ Solution.

**Roadmap**
1. What MCP is & why it exists
2. Architecture: host, client, server
3. The three primitives: tools, resources, prompts
4. Building an MCP server (tools)
5. Tool schemas & discovery
6. Resources (exposing data)
7. Prompts (reusable templates)
8. The client: discovering & calling
9. Wiring MCP into an agent
10. 🏆 Capstone: a complete MCP server + agent


In [ ]:
import json, re
from typing import Callable

print("We'll build MCP primitives from scratch: tools, resources, prompts.")

---
## Chapter 1 — What MCP Is & Why It Exists

📖 **Theory.** Before MCP, every LLM app integrated each tool/data source with **custom glue
code** — an N×M explosion (N apps × M integrations). MCP standardizes the interface: build one
MCP **server** for a capability, and it works with **any** MCP-compatible client. It's an open
protocol (JSON-RPC based) introduced by Anthropic and adopted widely.

🖼️ **Diagram — before vs after MCP**
```
 BEFORE (N×M glue):            AFTER (MCP standard):
  App1─┬─GitHub                 App1─┐         ┌─GitHub server
  App1─┼─Slack                  App2─┼─ MCP ───┼─Slack server
  App2─┼─GitHub                 App3─┘  (one   └─DB server
  App2─┴─DB                            protocol)
  ...every pair hand-built            ...build each server once
```

🧠 **Mental model.** MCP is a **universal adapter**. Write the adapter once per capability; plug
it into anything that speaks MCP.


In [ ]:
# The core promise: one standard interface for tools, data, and prompts.
mcp_primitives = {
    "tools":     "functions the model can CALL (actions/computation)",
    "resources": "data the model can READ (files, DB rows, API results)",
    "prompts":   "reusable prompt TEMPLATES the server offers",
}
for k, v in mcp_primitives.items():
    print(f"{k:10s}: {v}")

### ✏️ Your Turn 1.1
In a comment, explain the "N×M problem" in your own words and how MCP turns it into "N+M".

In [ ]:
# N×M problem: ...
# MCP makes it N+M because: ...


✅ **Solution**
```python
# N×M: N apps each need custom code for M integrations -> N*M bespoke connectors.
# MCP: each app speaks MCP once (N) and each integration is one server (M) -> N+M total.
```

---
## Chapter 2 — Architecture: Host, Client, Server

📖 **Theory.** Three roles:
- **Host** — the LLM application the user interacts with (Claude Desktop, an IDE, your agent).
- **Client** — lives inside the host; maintains a 1:1 connection to a server, handling the protocol.
- **Server** — a separate program exposing tools/resources/prompts for one capability.

🖼️ **Diagram — the MCP connection**
```
 ┌──────────── HOST (LLM app) ────────────┐
 │   ┌────────┐      ┌────────┐            │        ┌──────────────┐
 │   │ client │◄────►│ client │◄───────────┼───────►│ MCP server B │
 │   └───┬────┘      └────────┘  (JSON-RPC)│        └──────────────┘
 │       │                                 │        ┌──────────────┐
 │       └─────────────────────────────────┼───────►│ MCP server A │
 └─────────────────────────────────────────┘        └──────────────┘
```

🧠 **Mental model.** One **client per server** (a dedicated phone line), all managed by the host.
Servers are isolated — a crash or exploit in one doesn't touch the others.


In [ ]:
class MCPServer:
    """A minimal MCP server: holds tools, resources, and prompts."""
    def __init__(self, name):
        self.name = name
        self._tools, self._resources, self._prompts = {}, {}, {}

class MCPClient:
    """A minimal client: connects to one server and speaks its 'protocol'."""
    def __init__(self, server): self.server = server
    def ping(self): return f"connected to '{self.server.name}'"

server = MCPServer("demo-server")
client = MCPClient(server)
print(client.ping())

### ✏️ Your Turn 2.1
Create a second server named `"github-server"` and a client for it; confirm the client reports the
right server name.

In [ ]:
# make github server + client, ping it


✅ **Solution**
```python
gh = MCPServer("github-server")
gh_client = MCPClient(gh)
print(gh_client.ping())   # connected to 'github-server'
```

---
## Chapter 3 — The Three Primitives

📖 **Theory.** MCP servers expose exactly three kinds of capability:
- **Tools** — model-callable functions with side effects/computation (e.g. `create_issue`).
- **Resources** — readable data addressed by URI (e.g. `file:///readme.md`, `db://users/42`).
- **Prompts** — parameterized prompt templates the server provides (e.g. a "summarize PR" prompt).

🖼️ **Diagram — server surface**
```
 MCP server
 ├── tools:     [ create_issue, run_query, send_message ]   (actions)
 ├── resources: [ file://..., db://..., api://... ]         (data to read)
 └── prompts:   [ "summarize_pr", "triage_ticket" ]          (templates)
```

🧠 **Mental model.** **Tools = verbs** (do something), **Resources = nouns** (read something),
**Prompts = recipes** (a templated way to ask).


In [ ]:
# A quick taste of all three (we'll build each properly next)
def demo():
    server._tools["echo"] = {"fn": lambda text: text, "desc": "echo the input"}
    server._resources["motd://today"] = "Welcome to MCP!"
    server._prompts["greet"] = "Say hello to {name} warmly."
    return list(server._tools), list(server._resources), list(server._prompts)

print("tools/resources/prompts:", demo())

### ✏️ Your Turn 3.1
Classify each as tool, resource, or prompt (comment): (a) "delete a file", (b) "the contents of
config.json", (c) "a template that asks the model to write release notes".

In [ ]:
# (a) ...
# (b) ...
# (c) ...


✅ **Solution**
```python
# (a) tool     (an action with a side effect)
# (b) resource (data to read)
# (c) prompt   (a reusable template)
```

---
## Chapter 4 — Building an MCP Server (Tools)

📖 **Theory.** A tool needs a **name**, a **description**, an **input schema** (so the client/LLM
knows the arguments), and the **function**. Servers usually register tools with a decorator. Let's
build that ergonomics.

🖼️ **Diagram — registering a tool**
```
 @server.tool("add", "add two numbers", {"a":"number","b":"number"})
 def add(a, b): return a + b
        │
        ▼  stored in server._tools["add"]
```


In [ ]:
class MCPServer(MCPServer):   # extend with a tool decorator
    def tool(self, name, description, input_schema):
        def register(fn):
            self._tools[name] = {"fn": fn, "description": description, "schema": input_schema}
            return fn
        return register
    def call_tool(self, name, arguments: dict):
        if name not in self._tools:
            return {"error": f"unknown tool: {name}"}
        try:
            result = self._tools[name]["fn"](**arguments)
            return {"result": result}
        except Exception as e:
            return {"error": str(e)}

srv = MCPServer("math-server")

@srv.tool("add", "add two numbers", {"a": "number", "b": "number"})
def add(a, b): return a + b

@srv.tool("multiply", "multiply two numbers", {"a": "number", "b": "number"})
def multiply(a, b): return a * b

print(srv.call_tool("add", {"a": 3, "b": 4}))
print(srv.call_tool("multiply", {"a": 6, "b": 7}))
print(srv.call_tool("divide", {"a": 1, "b": 0}))   # unknown tool -> error

⚠️ **Common trap.** Always return errors as **structured data**, never crash the server. A
client calling an unknown tool or passing bad args should get a clean error object it can handle.

### ✏️ Your Turn 4.1
Register a `subtract` tool on `srv` and call it with `{"a": 10, "b": 3}`.

In [ ]:
# @srv.tool("subtract", ...) then call it


✅ **Solution**
```python
@srv.tool("subtract", "subtract b from a", {"a":"number","b":"number"})
def subtract(a, b): return a - b
print(srv.call_tool("subtract", {"a": 10, "b": 3}))   # {"result": 7}
```

---
## Chapter 5 — Tool Schemas & Discovery

📖 **Theory.** A client must **discover** what a server offers without reading its source. MCP
servers answer a "list tools" request returning each tool's name, description, and input schema —
which the host feeds to the LLM so it knows what it can call.

🖼️ **Diagram — discovery handshake**
```
 client ──"list_tools"──► server
 client ◄──[{name, description, schema}, ...]── server
        └─► host gives this list to the LLM as its available actions
```


In [ ]:
class MCPServer(MCPServer):
    def list_tools(self):
        return [{"name": n, "description": t["description"], "input_schema": t["schema"]}
                for n, t in self._tools.items()]

srv2 = MCPServer("math-server")
for name, (desc, sch) in {"add": ("add two numbers", {"a":"number","b":"number"})}.items():
    srv2._tools[name] = {"fn": lambda a,b: a+b, "description": desc, "schema": sch}

print(json.dumps(srv2.list_tools(), indent=2))

⚡ **Pro tip.** The **description** and **schema** are the model's only clues about a tool.
Invest in them: state what the tool does, when to use it, and exactly what each argument means.
This is the same lesson as the Agents lab — good descriptions drive correct tool use.

### ✏️ Your Turn 5.1
Call `list_tools()` on `srv` (from Ch.4, which has add/multiply/subtract) and print how many tools
it exposes and their names.

In [ ]:
# tools = srv.list_tools(); print count and names


✅ **Solution**
```python
tools = srv.list_tools()
print(len(tools), [t["name"] for t in tools])
```

---
## Chapter 6 — Resources (exposing data)

📖 **Theory.** **Resources** are readable data identified by a **URI**. Unlike tools (which
*act*), resources let the model *read* context: file contents, database records, API responses.
Clients list available resources and read them by URI.

🖼️ **Diagram — resource addressing**
```
 file:///docs/policy.md   ─► "Returns accepted within 30 days..."
 db://customers/42        ─► {"id":42,"name":"Ada","tier":"pro"}
 api://weather/paris      ─► {"temp":24,"sky":"clear"}
```


In [ ]:
class MCPServer(MCPServer):
    def resource(self, uri):
        def register(fn):
            self._resources[uri] = fn
            return fn
        return register
    def list_resources(self): return list(self._resources.keys())
    def read_resource(self, uri):
        if uri not in self._resources: return {"error": f"unknown resource: {uri}"}
        return {"uri": uri, "content": self._resources[uri]()}

docs = MCPServer("docs-server")

@docs.resource("file:///policy.md")
def policy(): return "Returns are accepted within 30 days with a receipt."

@docs.resource("db://customers/42")
def customer(): return {"id": 42, "name": "Ada", "tier": "pro"}

print("resources:", docs.list_resources())
print(docs.read_resource("file:///policy.md"))
print(docs.read_resource("db://customers/42"))

### ✏️ Your Turn 6.1
Add a resource `config://app` that returns a small config dict (e.g. `{"theme":"dark",
"lang":"en"}`), then read it.

In [ ]:
# register config://app resource and read it


✅ **Solution**
```python
@docs.resource("config://app")
def cfg(): return {"theme": "dark", "lang": "en"}
print(docs.read_resource("config://app"))
```

---
## Chapter 7 — Prompts (reusable templates)

📖 **Theory.** MCP servers can offer **prompt templates** — named, parameterized prompts the host
can present to users (e.g. a slash-command) or feed to the model. This lets a server ship *expert
prompts* alongside its tools/data.

🖼️ **Diagram — a served prompt**
```
 server prompt "summarize_pr"(files, author)
     │ fill args
     ▼
 "Summarize the PR by {author} touching {files} in 3 bullets."
```


In [ ]:
class MCPServer(MCPServer):
    def prompt(self, name):
        def register(fn):
            self._prompts[name] = fn
            return fn
        return register
    def list_prompts(self): return list(self._prompts.keys())
    def get_prompt(self, name, **kwargs):
        if name not in self._prompts: return {"error": f"unknown prompt: {name}"}
        return {"name": name, "text": self._prompts[name](**kwargs)}

ph = MCPServer("prompt-server")

@ph.prompt("triage_ticket")
def triage(ticket): return f"Classify this ticket and assign priority:\n{ticket}"

@ph.prompt("summarize_pr")
def sumpr(author, files): return f"Summarize the PR by {author} touching {files} in 3 bullets."

print("prompts:", ph.list_prompts())
print(ph.get_prompt("triage_ticket", ticket="I was charged twice"))
print(ph.get_prompt("summarize_pr", author="Ada", files="api.py, db.py"))

### ✏️ Your Turn 7.1
Add a `write_email` prompt taking `recipient` and `topic`, then render it.

In [ ]:
# register write_email prompt and get it


✅ **Solution**
```python
@ph.prompt("write_email")
def wr(recipient, topic): return f"Write a concise email to {recipient} about {topic}."
print(ph.get_prompt("write_email", recipient="the team", topic="the launch"))
```

---
## Chapter 8 — The Client: Discovering & Calling

📖 **Theory.** The **client** ties it together: it connects to a server, **discovers**
capabilities (list tools/resources/prompts), and **invokes** them on behalf of the host/LLM.

🖼️ **Diagram — client lifecycle**
```
 connect ─► discover (list_*) ─► expose to LLM ─► call_tool / read_resource / get_prompt
```


In [ ]:
class MCPClient(MCPClient):   # extend with full protocol methods
    def discover(self):
        return {
            "tools": self.server.list_tools() if hasattr(self.server, "list_tools") else [],
            "resources": self.server.list_resources() if hasattr(self.server, "list_resources") else [],
            "prompts": self.server.list_prompts() if hasattr(self.server, "list_prompts") else [],
        }
    def call_tool(self, name, arguments): return self.server.call_tool(name, arguments)
    def read_resource(self, uri): return self.server.read_resource(uri)

# a server with all three primitives
full = MCPServer("all-in-one")
@full.tool("add", "add two numbers", {"a":"number","b":"number"})
def _add(a, b): return a + b
@full.resource("file:///readme.md")
def _readme(): return "This server does math."
@full.prompt("explain")
def _explain(topic): return f"Explain {topic} simply."

c = MCPClient(full)
print("discovered:", json.dumps(c.discover(), indent=2))
print("call add:", c.call_tool("add", {"a": 20, "b": 22}))

⚠️ **Common trap.** Discovery and invocation are **separate steps**. The host discovers *once*
(to tell the LLM what's available), then invokes *per request*. Don't re-discover on every call —
cache the capability list.

### ✏️ Your Turn 8.1
Use client `c` to read the `file:///readme.md` resource and print its content.

In [ ]:
# c.read_resource(...)


✅ **Solution**
```python
print(c.read_resource("file:///readme.md")["content"])
```

---
## Chapter 9 — Wiring MCP into an Agent

📖 **Theory.** MCP is the **plumbing** that gives an agent its tools. The agent discovers tools via
the client, the LLM picks one, and the client invokes it on the server. MCP + the agent loop
(from the Agents lab) = a tool-using assistant with standardized, swappable capabilities.

🖼️ **Diagram — MCP-powered agent**
```
 goal ─► agent ─► (LLM picks tool from client.discover()) ─► client.call_tool ─► server ─► result ─► answer
```


In [ ]:
def mcp_agent(goal, client):
    tools = {t["name"]: t for t in client.discover()["tools"]}
    # naive routing (a real agent lets the LLM choose from descriptions)
    m = re.search(r"(\d+)\s*\+\s*(\d+)", goal)
    if m and "add" in tools:
        a, b = int(m.group(1)), int(m.group(2))
        out = client.call_tool("add", {"a": a, "b": b})
        return f"[add] {out['result']}"
    return "no matching tool"

print(mcp_agent("what is 20 + 22?", c))

### ✏️ Your Turn 9.1
Add a `multiply` tool to the `full` server, then extend `mcp_agent` to handle `"6 * 7"` style
goals via the client.

In [ ]:
# add multiply tool to `full`, extend mcp_agent for "*"


✅ **Solution**
```python
@full.tool("multiply", "multiply two numbers", {"a":"number","b":"number"})
def _mul(a, b): return a * b
# in mcp_agent, add:
# m = re.search(r"(\d+)\s*\*\s*(\d+)", goal)
# if m and "multiply" in tools: ... client.call_tool("multiply", {...})
```

---
## 🏆 Chapter 10 — Capstone: A Complete MCP Server + Agent

Build a **`WeatherServer`** (an MCP server) exposing: a `get_weather` **tool**, a
`weather://cities` **resource** listing supported cities, and a `weather_report` **prompt**
template. Then drive it through a client with a small agent. Build it before revealing the solution.

In [ ]:
# Your WeatherServer + client + agent here
weather_server = MCPServer("weather-server")

# register a get_weather tool, a weather://cities resource, and a weather_report prompt

# wclient = MCPClient(weather_server)
# print(wclient.discover())
# print(wclient.call_tool("get_weather", {"city": "Paris"}))


✅ **Capstone Solution**
```python
weather_server = MCPServer("weather-server")
_DB = {"paris": "Sunny 24C", "london": "Rainy 15C", "tokyo": "Cloudy 20C"}

@weather_server.tool("get_weather", "current weather for a city", {"city": "string"})
def get_weather(city):
    return _DB.get(city.lower(), "unavailable")

@weather_server.resource("weather://cities")
def cities():
    return list(_DB.keys())

@weather_server.prompt("weather_report")
def weather_report(city):
    return f"Give a friendly one-line weather report for {city}."

wclient = MCPClient(weather_server)
print("discover:", json.dumps(wclient.discover(), indent=2))
print("cities:", wclient.read_resource("weather://cities")["content"])
print("weather:", wclient.call_tool("get_weather", {"city": "Tokyo"}))
print("prompt:", weather_server.get_prompt("weather_report", city="Paris"))

# a tiny agent using the MCP client
def weather_agent(goal, client):
    tools = {t["name"]: t for t in client.discover()["tools"]}
    if "weather" in goal.lower() and "get_weather" in tools:
        city = goal.lower().split("in")[-1].strip(" ?.")
        return client.call_tool("get_weather", {"city": city})["result"]
    return "no matching tool"

print(weather_agent("what's the weather in London?", wclient))
```

🎉 **You understand MCP end to end!** Host/client/server architecture, the three primitives
(tools, resources, prompts), discovery, invocation, and wiring MCP into an agent. Real MCP adds
JSON-RPC transport (stdio/HTTP) and auth, but every concept maps to what you built. Next step: the
official SDK (`pip install mcp`) exposes these exact primitives with `@server.tool()` decorators.

---
### 📌 Concept Quick-Reference
**Why MCP:** standard interface turns N×M custom integrations into N+M
**Architecture:** host (LLM app) ▸ client (1 per server) ▸ server (a capability)
**Primitives:** tools (actions/verbs), resources (readable data/nouns), prompts (templates/recipes)
**Server:** register tools/resources/prompts; return structured errors, never crash
**Discovery:** list_tools/list_resources/list_prompts → given to the LLM
**Invocation:** call_tool(name, args), read_resource(uri), get_prompt(name, **args)
**Agent wiring:** discover → LLM picks tool → client.call_tool → server → result
**Real SDK:** `pip install mcp`; transports stdio/HTTP + JSON-RPC + auth
